<a href="https://colab.research.google.com/github/Hotmandooo/-zero.ipynb/blob/main/%EA%B7%B8%EB%9E%98%ED%94%84_%EA%B3%BC%EC%A0%9C_%EC%95%88%EC%9A%B0%ED%98%84%EC%A1%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [09] 그래프 실습 과제 — DFS, BFS, 감염 시뮬레이션

**조원:** 안우현, 이가을, 박채완, 정민서, 윤여주, 권태현 (총 6명)

| 항목 | 내용 |
| :--- | :--- |
| **사용 AI 서비스** | Google Gemini (바이브 코딩 및 알고리즘·시각화·검증 코드 설계) |
| **바이브 코딩 범위** | 1~7번 전체 노트북 구조 설계, NetworkX 레벨별 시각화, 2단계 감염 경로 역추적 시뮬레이터, 테스트 케이스(하·하·중·중·중) 생성 및 자동 검증(`check`) 함수 |
| **사용 프롬프트** | 그래프 실습 PDF 마지막 장 과제를 구글 코랩(`.ipynb`)에서 실행할 수 있게 만들어줘. 우리 조 그래프는 안우현: 최우진, 배정우, 김서준, 이서아, 박도윤 / 이가을: 김준석, 강석민, 김진우, 소준영, 김유진 / 박채완: 이도현, 이대환, 오수아, 차재욱, 조형철 / 정민서: 이준수, 김준성, 최하은, 정시우, 강유나 / 윤여주: 김수인, 황정민, 김민정, 정다연, 안서윤 / 권태현: 김주홍, 염도형, 김기헌, 박진성, 나희재 야. 팀원끼리는 안우현 → 이가을 → 박채완 → 정민서 → 윤여주 → 권태현 순서로 연결하고, 하·하·중·중·중 테스트 케이스와 검증 코드까지 상세 주석과 함께 넣어줘. |

## 1. 그래프 구성 (인접 리스트)

* 수업 예시처럼 **방향 그래프 (`A: [B]` = `A → B`)** 형태의 파이썬 딕셔너리(인접 리스트)로 구성했습니다.
* 우리 조원 6명이 하나의 네트워크로 이어져 DFS·BFS 및 다단계 감염 전파를 확인할 수 있도록 **팀원끼리는 `안우현 → 이가을 → 박채완 → 정민서 → 윤여주 → 권태현` 순서로 연결**했습니다. (각 팀원 리스트의 첫 번째에 다음 팀원을 배치)
* 36명 전원이 고유한 이름(동명이인 없음)이며, 더 이상 연결할 이웃이 없는 친구 노드는 빈 리스트(`[]`)로 두어 탐색 시 `KeyError`가 발생하지 않도록 구성했습니다.
* **총 노드 수:** 팀원 6명 + 친구 30명 = **36개** / **총 간선 수:** 팀원 연결 5개 + 친구 연결 30개 = **35개**

In [ ]:
# [1단계] 우리 조 인접 리스트(딕셔너리) 구성
# - 키(Key): 전파를 시작하는 사람(출발 노드)
# - 값(Value): 그 사람이 직접 접촉한 사람 목록(도착 노드 리스트)
# - 각 팀원 리스트의 맨 앞(0번 인덱스)에 다음 팀원을 배치하여 팀원 6명이 사슬처럼 연결됨
graph = {
    '안우현': ['이가을', '최우진', '배정우', '김서준', '이서아', '박도윤'],  # 시작 팀원(Level 0)
    '이가을': ['박채완', '김준석', '강석민', '김진우', '소준영', '김유진'],  # 2번째 팀원
    '박채완': ['정민서', '이도현', '이대환', '오수아', '차재욱', '조형철'],  # 3번째 팀원
    '정민서': ['윤여주', '이준수', '김준성', '최하은', '정시우', '강유나'],  # 4번째 팀원
    '윤여주': ['권태현', '김수인', '황정민', '김민정', '정다연', '안서윤'],  # 5번째 팀원
    '권태현': ['김주홍', '염도형', '김기헌', '박진성', '나희재'],            # 마지막 팀원

    # 아래 30명은 각 팀원의 친구(리프 노드)이며, 더 이상 퍼뜨릴 다음 대상이 없으므로 빈 리스트([])로 설정
    '최우진': [], '배정우': [], '김서준': [], '이서아': [], '박도윤': [],
    '김준석': [], '강석민': [], '김진우': [], '소준영': [], '김유진': [],
    '이도현': [], '이대환': [], '오수아': [], '차재욱': [], '조형철': [],
    '이준수': [], '김준성': [], '최하은': [], '정시우': [], '강유나': [],
    '김수인': [], '황정민': [], '김민정': [], '정다연': [], '안서윤': [],
    '김주홍': [], '염도형': [], '김기헌': [], '박진성': [], '나희재': [],
}

# 전체 정점(노드) 수와 간선(화살표) 수 확인
print('노드 수:', len(graph))
print('간선 수:', sum(len(v) for v in graph.values()))

노드 수: 36
간선 수: 35


## 2. DFS 탐색 (재귀)

* **깊이 우선 탐색(DFS)**은 스택(재귀 호출)을 사용하여 **갈 수 있는 한 경로의 끝까지 깊숙이 파고든 뒤**, 막다른 길에 다다르면 직전 갈림길로 되돌아와(Backtracking) 나머지 이웃을 탐색합니다.
* 각 팀원 리스트의 맨 앞에 다음 팀원이 있으므로, `안우현`에서 시작하면 자신의 직접 친구(`최우진` 등)보다 **팀원 사슬 끝(`권태현`과 그 친구들)을 먼저 방문**하게 됩니다.

In [ ]:
def dfs(graph, v, visited, order=None):
    # [준비] 방문 순서를 기록할 리스트가 없으면 새로 생성
    if order is None:
        order = []

    visited.add(v)       # 현재 도착한 사람(v)을 방문 완료 집합에 등록 (중복 방문 방지)
    order.append(v)      # 전체 방문 순서 기록 리스트에 추가
    print(v, end=' ')

    # [핵심 반복] 현재 사람(v)과 연결된 이웃(neighbor)들을 앞에서부터 순서대로 하나씩 확인
    # -> 목표: 아직 방문하지 않은 이웃을 만나면 즉시 그 사람의 재귀 함수로 깊숙이 들어가기 위함
    for neighbor in graph[v]:
        if neighbor not in visited:
            dfs(graph, neighbor, visited, order)  # 막다른 길이 나올 때까지 재귀 호출로 끝까지 파고듦

    return order

print("DFS 탐색 순서:")
dfs_order = dfs(graph, '안우현', set())
print()

DFS 탐색 순서:
안우현 이가을 박채완 정민서 윤여주 권태현 김주홍 염도형 김기헌 박진성 나희재 김수인 황정민 김민정 정다연 안서윤 이준수 김준성 최하은 정시우 강유나 이도현 이대환 오수아 차재욱 조형철 김준석 강석민 김진우 소준영 김유진 최우진 배정우 김서준 이서아 박도윤 


## 3. BFS 탐색 (큐)

* **너비 우선 탐색(BFS)**은 선입선출(FIFO) 구조의 **큐(`collections.deque`)**를 사용하여, 시작 노드로부터 **거리가 가까운 1단계 이웃들을 모두 먼저 방문한 뒤** 2단계, 3단계 순서로 물결처럼 퍼져 나갑니다.

In [ ]:
from collections import deque

def bfs(graph, start):
    visited = set()              # 이미 큐에 넣었거나 방문한 사람 집합 (중복 삽입 방지)
    queue = deque([start])       # 앞으로 방문할 사람을 순서대로 줄 세우는 대기열(큐)
    visited.add(start)           # 시작 노드를 큐에 넣으면서 동시에 방문 도장을 찍음
    order = []                   # 최종 BFS 방문 순서를 저장할 리스트

    # [핵심 반복] 대기열(queue)에 사람이 남아 있는 동안 맨 앞에서 한 명씩 꺼내며 탐색
    # -> 목표: 가까운 거리(레벨)에 있는 사람부터 빠짐없이 순서대로 방문하기 위함
    while queue:
        v = queue.popleft()      # 대기열 맨 앞사람(가장 먼저 줄 선 사람)을 꺼냄
        order.append(v)
        print(v, end=' ')

        # 현재 사람(v)의 모든 이웃을 확인하여 아직 방문하지 않은 사람만 대기열 맨 뒤에 줄 세움
        for neighbor in graph[v]:
            if neighbor not in visited:
                visited.add(neighbor)      # 큐에 넣는 즉시 방문 처리하여 중복 줄서기 차단
                queue.append(neighbor)

    return order

print("BFS 탐색 순서:")
bfs_order = bfs(graph, '안우현')
print()

BFS 탐색 순서:
안우현 이가을 최우진 배정우 김서준 이서아 박도윤 박채완 김준석 강석민 김진우 소준영 김유진 정민서 이도현 이대환 오수아 차재욱 조형철 윤여주 이준수 김준성 최하은 정시우 강유나 권태현 김수인 황정민 김민정 정다연 안서윤 김주홍 염도형 김기헌 박진성 나희재 


## 4. 손으로 써 본 탐색 순서와 비교 (시작: `안우현`)

### ① DFS (손 계산)
다음 팀원이 리스트 맨 앞(`0`번 인덱스)에 있으므로 팀원 사슬의 끝(`권태현`)까지 먼저 내려간 뒤, 거꾸로 올라오면서 각 팀원의 친구들을 방문합니다.

> `안우현 → 이가을 → 박채완 → 정민서 → 윤여주 → 권태현 → 김주홍 → 염도형 → 김기헌 → 박진성 → 나희재 → (윤여주로 복귀) 김수인 → 황정민 → 김민정 → 정다연 → 안서윤 → (정민서로 복귀) 이준수 → 김준성 → 최하은 → 정시우 → 강유나 → (박채완으로 복귀) 이도현 → 이대환 → 오수아 → 차재욱 → 조형철 → (이가을로 복귀) 김준석 → 강석민 → 김진우 → 소준영 → 김유진 → (안우현으로 복귀) 최우진 → 배정우 → 김서준 → 이서아 → 박도윤`

### ② BFS (손 계산 — 레벨별 정리)
시작점(`안우현`)으로부터 떨어진 거리(레벨) 순서대로 층층이 방문합니다.

| 레벨 (거리) | 방문 노드 (총 36명) | 인원 |
| :---: | :--- | :---: |
| **0** | `안우현` | 1명 |
| **1** | `이가을, 최우진, 배정우, 김서준, 이서아, 박도윤` | 6명 |
| **2** | `박채완, 김준석, 강석민, 김진우, 소준영, 김유진` | 6명 |
| **3** | `정민서, 이도현, 이대환, 오수아, 차재욱, 조형철` | 6명 |
| **4** | `윤여주, 이준수, 김준성, 최하은, 정시우, 강유나` | 6명 |
| **5** | `권태현, 김수인, 황정민, 김민정, 정다연, 안서윤` | 6명 |
| **6** | `김주홍, 염도형, 김기헌, 박진성, 나희재` | 5명 |

### 비교 결론
* **DFS**는 한 방향으로 끝까지 깊이 들어가므로 `안우현`의 직접 친구인 `최우진`을 **32번째**에야 방문합니다.
* **BFS**는 가까운 노드부터 방문하므로 `최우진`을 **3번째**로 방문하고, 가장 먼 `권태현`의 친구들을 마지막에 방문합니다.
* 두 알고리즘 모두 36개 노드를 빠짐없이 한 번씩 방문하지만, **"가까운 접촉자부터 단계별로 추적"**해야 하는 감염 시뮬레이션에는 **BFS가 압도적으로 적합**합니다.

In [ ]:
# 손으로 직접 계산한 DFS / BFS 탐색 순서 리스트
hand_dfs = [
    '안우현', '이가을', '박채완', '정민서', '윤여주', '권태현',
    '김주홍', '염도형', '김기헌', '박진성', '나희재',
    '김수인', '황정민', '김민정', '정다연', '안서윤',
    '이준수', '김준성', '최하은', '정시우', '강유나',
    '이도현', '이대환', '오수아', '차재욱', '조형철',
    '김준석', '강석민', '김진우', '소준영', '김유진',
    '최우진', '배정우', '김서준', '이서아', '박도윤'
]

hand_bfs = [
    '안우현',
    '이가을', '최우진', '배정우', '김서준', '이서아', '박도윤',
    '박채완', '김준석', '강석민', '김진우', '소준영', '김유진',
    '정민서', '이도현', '이대환', '오수아', '차재욱', '조형철',
    '윤여주', '이준수', '김준성', '최하은', '정시우', '강유나',
    '권태현', '김수인', '황정민', '김민정', '정다연', '안서윤',
    '김주홍', '염도형', '김기헌', '박진성', '나희재'
]

# 손 계산 결과와 파이썬 실행 결과가 100% 일치하는지 검증
print('DFS 손 계산 == 코드 결과 :', hand_dfs == dfs_order)
print('BFS 손 계산 == 코드 결과 :', hand_bfs == bfs_order)
print()

# 1번부터 36번까지 DFS와 BFS 방문 순서 나란히 비교 출력
print(f"{'순번':>4} | {'DFS':<6} | {'BFS':<6}")
print('-' * 26)
for i, (d, b) in enumerate(zip(dfs_order, bfs_order), 1):
    print(f"{i:>4} | {d:<6} | {b:<6}")

DFS 손 계산 == 코드 결과 : True
BFS 손 계산 == 코드 결과 : True

  순번 | DFS    | BFS   
--------------------------
   1 | 안우현    | 안우현   
   2 | 이가을    | 이가을   
   3 | 박채완    | 최우진   
   4 | 정민서    | 배정우   
   5 | 윤여주    | 김서준   
   6 | 권태현    | 이서아   
   7 | 김주홍    | 박도윤   
   8 | 염도형    | 박채완   
   9 | 김기헌    | 김준석   
  10 | 박진성    | 강석민   
  11 | 나희재    | 김진우   
  12 | 김수인    | 소준영   
  13 | 황정민    | 김유진   
  14 | 김민정    | 정민서   
  15 | 정다연    | 이도현   
  16 | 안서윤    | 이대환   
  17 | 이준수    | 오수아   
  18 | 김준성    | 차재욱   
  19 | 최하은    | 조형철   
  20 | 정시우    | 윤여주   
  21 | 강유나    | 이준수   
  22 | 이도현    | 김준성   
  23 | 이대환    | 최하은   
  24 | 오수아    | 정시우   
  25 | 차재욱    | 강유나   
  26 | 조형철    | 권태현   
  27 | 김준석    | 김수인   
  28 | 강석민    | 황정민   
  29 | 김진우    | 김민정   
  30 | 소준영    | 정다연   
  31 | 김유진    | 안서윤   
  32 | 최우진    | 김주홍   
  33 | 배정우    | 염도형   
  34 | 김서준    | 김기헌   
  35 | 이서아    | 박진성   
  36 | 박도윤    | 나희재   


## 5. (선택) NetworkX 시각화

Colab 환경에서 한글이 깨지지 않도록 `나눔고딕(NanumGothic)` 폰트를 설치한 뒤, 시작 노드(`안우현`)로부터의 최단 거리(레벨 `0 ~ 6`)에 따라 왼쪽에서 오른쪽으로 층을 나누어 시각화합니다.

In [ ]:
# 한글 폰트 설치 (Google Colab 전용 - 처음 한 번만 실행)
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

def bfs_levels(graph, start):
    """시작 노드(start)로부터 각 노드까지의 최단 거리(레벨)를 계산하는 함수"""
    level = {start: 0}
    queue = deque([start])

    # [핵심 반복] 큐에서 노드를 꺼내 아직 거리가 기록되지 않은 이웃에게 (현재 거리 + 1)을 부여
    while queue:
        v = queue.popleft()
        for w in graph.get(v, []):
            if w not in level:
                level[w] = level[v] + 1
                queue.append(w)
    return level

def draw_graph(graph, start, highlight=None, title=''):
    """레벨별로 X좌표를 배치하고 색상을 구분해 NetworkX 방향 그래프를 그리는 함수"""
    G = nx.DiGraph()
    for v, nbrs in graph.items():
        G.add_node(v)
        for w in nbrs:
            G.add_edge(v, w)

    level = bfs_levels(graph, start)

    # [좌표 배치 반복] 레벨(거리)을 X좌표로, 같은 레벨 안의 순서를 Y좌표로 균등 배분하여 겹침 방지
    by_level = {}
    for v in G.nodes:
        by_level.setdefault(level.get(v, -1), []).append(v)

    pos = {}
    for lv, nodes in by_level.items():
        for i, v in enumerate(nodes):
            pos[v] = (lv, -(i - (len(nodes) - 1) / 2))

    # 레벨별 색상 팔레트 (0: 빨강, 1: 주황, 2: 노랑, 3: 하늘, 4: 연두, 5: 보라, 6: 분홍)
    palette = ['#e63946', '#f4a261', '#e9c46a', '#a8dadc', '#90be6d', '#cdb4db', '#ffafcc']
    if highlight is None:
        colors = [palette[level[v] % len(palette)] if v in level else '#dddddd' for v in G.nodes]
    else:
        colors = [palette[highlight[v]] if v in highlight else '#dddddd' for v in G.nodes]

    plt.figure(figsize=(15, 8))
    nx.draw_networkx_edges(G, pos, arrows=True, edge_color='#999999', width=1.4, arrowsize=14, node_size=1800)
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=1800, edgecolors='#333333', linewidths=1.2)
    nx.draw_networkx_labels(G, pos, font_family='NanumGothic', font_size=9.5, font_weight='bold')
    plt.title(title, fontsize=15, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# 전체 36명 네트워크 시각화 실행 (Colab에서 위 폰트 설치 셀 실행 후 돌리면 한글이 선명하게 표시됩니다)
draw_graph(graph, '안우현', title='우리 조 친구 그래프 36명 (색상 = 안우현으로부터의 최단 거리)')

## 6. 감염 시뮬레이션 (BFS 확장) — 2단계 이내 전염 경로 추적

### 설계 아이디어
* 수업의 `recommend_friend`는 `depth == 2`인 노드만 출력했지만, 감염 시뮬레이션은 **1단계(직접 접촉) + 2단계(접촉자의 접촉자) 모두 출력**해야 하며 **최초 감염자 본인은 제외**해야 합니다.
* BFS는 가까운 노드부터 방문하므로, 어떤 노드를 처음 방문했을 때의 거리 = **최단 감염 거리(레벨)**가 보장됩니다.
* 각 노드가 **누구에게서 감염되었는지(`parent`)**를 함께 저장해 두면, 감염자부터 최초 감염자까지 거꾸로 따라가 **전염 경로(`안우현 → 이가을 → 김준석`)**를 정확히 복원할 수 있습니다.
* 큐에서 꺼낼 때가 아니라 **큐에 넣을 때 즉시 `parent`에 등록**하므로, 노드 1,000개 규모의 고밀도 그래프에서도 중복 큐 삽입 없이 빠르게 동작합니다.

In [ ]:
def bfs_levels(graph, start):
    """시각화 셀을 건너뛰고 실행해도 검증 함수(check)가 동작하도록 최단 거리 계산 함수 정의"""
    level = {start: 0}
    queue = deque([start])
    while queue:
        v = queue.popleft()
        for w in graph.get(v, []):
            if w not in level:
                level[w] = level[v] + 1
                queue.append(w)
    return level

def infection_simulation(graph, patient_zero, max_depth=2):
    """
    최초 감염자(patient_zero)로부터 max_depth 단계 이내의 감염자와 전염 경로를 BFS로 추적
    반환 형식: {감염된_사람: (감염_레벨, [최초감염자, ..., 감염된_사람])} (감염자 본인은 제외)
    """
    if patient_zero not in graph:
        raise ValueError(f"'{patient_zero}'은(는) 그래프에 없는 사람입니다.")

    parent = {patient_zero: None}           # 누가 누구를 감염시켰는지 기록 (방문 여부 체크 겸용)
    queue = deque([(patient_zero, 0)])      # (현재 조사할 감염자, 감염 거리) 대기열
    result = {}                             # 최종 결과 저장 딕셔너리

    # [핵심 반복 1] 대기열에서 한 명씩 꺼내며 max_depth 단계까지 전염을 확산시킴
    while queue:
        current, depth = queue.popleft()

        # 목표 단계(예: 2단계)에 도달한 사람은 더 이상 다음 사람에게 전파하지 않고 중단
        if depth == max_depth:
            continue

        # [핵심 반복 2] current와 직접 접촉한 이웃들을 확인하여 신규 감염자 등록 및 경로 복원
        for neighbor in graph.get(current, []):
            if neighbor not in parent:          # 처음 방문한 사람 = 최단 감염 거리 확정
                parent[neighbor] = current      # neighbor를 감염시킨 장본인(current) 기록

                # [경로 복원 반복] neighbor에서 parent를 거꾸로 타고 올라가 최초 감염자까지의 경로 생성
                path, node = [], neighbor
                while node is not None:
                    path.append(node)
                    node = parent[node]

                # 역순으로 모인 경로를 뒤집어(path[::-1]) '안우현 → ... → neighbor' 순서로 저장
                result[neighbor] = (depth + 1, path[::-1])
                queue.append((neighbor, depth + 1))

    return result

def print_infection(graph, patient_zero, max_depth=2, show_path=True, limit=None):
    """단계별(1단계, 2단계, ...) 감염자 명단과 전염 경로를 보기 좋게 출력하는 함수"""
    result = infection_simulation(graph, patient_zero, max_depth)
    print(f"[감염자: {patient_zero}] {max_depth}단계 이내 감염 예상 인원: {len(result)}명 (본인 제외)")

    for lv in range(1, max_depth + 1):
        members = [v for v, (d, _) in result.items() if d == lv]
        if not members:
            continue
        print(f"\n● {lv}단계 ({len(members)}명)")
        shown = members if limit is None else members[:limit]
        for v in shown:
            if show_path:
                print(f"  - {v:<6} 경로: {' → '.join(result[v][1])}")
            else:
                print(f"  - {v}")
        if limit is not None and len(members) > limit:
            print(f"  ... 외 {len(members) - limit}명")
    return result

# 우리 조 그래프에서 '안우현'이 최초 감염자일 때 2단계 이내 전파 시뮬레이션 실행
infected = print_infection(graph, '안우현', max_depth=2)

[감염자: 안우현] 2단계 이내 감염 예상 인원: 12명 (본인 제외)

● 1단계 (6명)
  - 이가을    경로: 안우현 → 이가을
  - 최우진    경로: 안우현 → 최우진
  - 배정우    경로: 안우현 → 배정우
  - 김서준    경로: 안우현 → 김서준
  - 이서아    경로: 안우현 → 이서아
  - 박도윤    경로: 안우현 → 박도윤

● 2단계 (6명)
  - 박채완    경로: 안우현 → 이가을 → 박채완
  - 김준석    경로: 안우현 → 이가을 → 김준석
  - 강석민    경로: 안우현 → 이가을 → 강석민
  - 김진우    경로: 안우현 → 이가을 → 김진우
  - 소준영    경로: 안우현 → 이가을 → 소준영
  - 김유진    경로: 안우현 → 이가을 → 김유진


In [ ]:
# 감염 결과 시각화: 감염자(빨강) / 1단계(주황) / 2단계(노랑) / 미감염(회색)
highlight = {'안우현': 0, **{v: d for v, (d, _) in infected.items()}}
draw_graph(graph, '안우현', highlight=highlight, title='안우현 감염 시 2단계 이내 전파 범위 (빨강: 감염자, 주황: 1단계, 노랑: 2단계)')

## 7. 테스트 케이스 및 자동 무결성 검증 (`check` 함수)

과제 요구사항에 맞추어 **난이도별 5종 테스트 케이스(하 · 하 · 중 · 중 · 중)**와 **예외 상황**을 구성하고, 모든 결과가 수학적·논리적 조건을 100% 만족하는지 `check()` 함수로 자동 검증합니다.

| 구분 | 케이스명 | 그래프 특징 | 노드 수 | Max Depth |
| :---: | :--- | :--- | :---: | :---: |
| **하 1** | 기본 2단계 + 다중 경로 | 우리 조 `이가을` 기준 기본 전파 및 경로 검증 | 36 | 2 |
| **하 2** | 역방향 순환 + 고립 집단 | 순환(`Cycle`) 무한 루프 방어 및 외부 고립 집단(2명) 안전 보호 | 11 | 2 |
| **중 1** | 표준 커뮤니티 | 100명 규모 네트워크에서 깊이 10까지 연결 후 2단계 조기 종료 검증 | 100 | 10 |
| **중 2** | 슈퍼 전파자 허브 | 최초 감염자 `정민서`에게 15명이 집중 연결된 폭발적 확산 시뮬레이션 | 100 | 10 |
| **중 3** | 고밀도 순환 + 방역 격리 | 고밀도 88명 + 외부 차단 격리 집단 12명 (2단계 & 10단계 끝까지 추적) | 100 | 10 |
| **예외** | 경계 조건 테스트 | 친구 리스트가 빈 리프 노드 / `max_depth=0` / 그래프에 없는 이름 예외 처리 | - | - |

In [ ]:
def check(graph, patient_zero, max_depth, result):
    """
    [자동 검증 함수] 시뮬레이션 결과가 아래 5가지 핵심 조건을 모두 통과하는지 assert로 검증
    1. 최초 감염자 본인이 결과에 포함되지 않았다.
    2. 모든 감염자의 레벨이 1 이상 max_depth 이하이다.
    3. 각 사람의 레벨이 제한 없는 BFS로 구한 실제 최단 거리와 정확히 일치한다.
    4. max_depth 이내에 도달 가능한 모든 노드가 누락 없이 포함되었다.
    5. 복원된 전염 경로가 최초 감염자에서 시작해 해당 노드로 끝나며, 경로의 모든 연결이 실제 간선이다.
    """
    true_level = bfs_levels(graph, patient_zero)
    assert patient_zero not in result, '오류: 감염자 본인이 결과에 포함됨'

    for v, (lv, path) in result.items():
        assert 1 <= lv <= max_depth, f'{v}: 레벨 범위 오류 ({lv})'
        assert lv == true_level[v], f'{v}: 최단 거리가 아님 ({lv} != {true_level[v]})'
        assert path[0] == patient_zero and path[-1] == v, f'{v}: 경로 시작/끝 불일치'
        assert len(path) - 1 == lv, f'{v}: 경로 길이 != 레벨'
        for a, b in zip(path, path[1:]):
            assert b in graph[a], f'{v}: 그래프에 없는 간선 {a}->{b}'

    expected = {v for v, d in true_level.items() if 1 <= d <= max_depth}
    assert set(result) == expected, '오류: 누락되거나 초과된 노드가 있음'
    print(f'✅ 검증 통과 (감염자: {patient_zero}, max_depth={max_depth}, 총 감염={len(result)}명)')

In [ ]:
# ==============================================================================
# [테스트 케이스 1 - 하1] 우리 조 그래프 (감염자: 이가을, max_depth=2)
# - 역할: 팀원 사슬(이가을 -> 박채완 -> 정민서)과 친구 그룹으로 2단계 전파 및 경로 복원 검증
# ==============================================================================
r_easy1 = print_infection(graph, '이가을', max_depth=2)
check(graph, '이가을', 2, r_easy1)

[감염자: 이가을] 2단계 이내 감염 예상 인원: 12명 (본인 제외)

● 1단계 (6명)
  - 박채완    경로: 이가을 → 박채완
  - 김준석    경로: 이가을 → 김준석
  - 강석민    경로: 이가을 → 강석민
  - 김진우    경로: 이가을 → 김진우
  - 소준영    경로: 이가을 → 소준영
  - 김유진    경로: 이가을 → 김유진

● 2단계 (6명)
  - 정민서    경로: 이가을 → 박채완 → 정민서
  - 이도현    경로: 이가을 → 박채완 → 이도현
  - 이대환    경로: 이가을 → 박채완 → 이대환
  - 오수아    경로: 이가을 → 박채완 → 오수아
  - 차재욱    경로: 이가을 → 박채완 → 차재욱
  - 조형철    경로: 이가을 → 박채완 → 조형철
✅ 검증 통과 (감염자: 이가을, max_depth=2, 총 감염=12명)


In [ ]:
# ==============================================================================
# [테스트 케이스 2 - 하2] 역방향 순환(Cycle) + 고립 집단 검증 (노드 11개, max_depth=2)
# - 역할: 꼬리를 무는 순환 구조에서 무한 루프 차단 및 외부와 단절된 고립 집단(2명) 안전 보호 확인
# ==============================================================================
graph_cycle_isolated = {
    '박채완': ['이도현', '이대환', '오수아'],
    '이도현': ['박채완', '차재욱', '조형철', '이대환'],  # 서로 주고받는 양방향 순환
    '이대환': ['조형철', '정민서', '이준수', '박채완'],
    '오수아': ['이준수', '김준성', '이도현'],
    '차재욱': ['박채완', '조형철'],                    # 2단계에서 0단계로 되돌아가는 역순환
    '조형철': ['이도현'],
    '정민서': [],
    '이준수': ['김준성'],
    '김준성': [],
    '윤여주': ['김수인'],   # [고립 집단] 박채완 쪽과 연결선이 전혀 없어 절대 감염되지 않음
    '김수인': ['윤여주'],
}

r_easy2 = print_infection(graph_cycle_isolated, '박채완', max_depth=2)
check(graph_cycle_isolated, '박채완', 2, r_easy2)
assert '윤여주' not in r_easy2 and '김수인' not in r_easy2, '고립 집단이 감염되면 안 됩니다!'
print('🛡️ 고립 집단(윤여주, 김수인 2명) 안전 보호 확인 완료!')

[감염자: 박채완] 2단계 이내 감염 예상 인원: 8명 (본인 제외)

● 1단계 (3명)
  - 이도현    경로: 박채완 → 이도현
  - 이대환    경로: 박채완 → 이대환
  - 오수아    경로: 박채완 → 오수아

● 2단계 (5명)
  - 차재욱    경로: 박채완 → 이도현 → 차재욱
  - 조형철    경로: 박채완 → 이도현 → 조형철
  - 정민서    경로: 박채완 → 이대환 → 정민서
  - 이준수    경로: 박채완 → 이대환 → 이준수
  - 김준성    경로: 박채완 → 오수아 → 김준성
✅ 검증 통과 (감염자: 박채완, max_depth=2, 총 감염=8명)
🛡️ 고립 집단(윤여주, 김수인 2명) 안전 보호 확인 완료!


In [ ]:
# ==============================================================================
# [테스트 케이스 생성기] 중 난이도(노드 100개, Max Depth 10) 그래프 생성 함수
# - 원리: 먼저 시작점부터 깊이 10까지 이어지는 기준 뼈대 경로를 세운 뒤,
#         기존 최단 거리를 단축시키지 않는 조건(depths[v] <= depths[u] + 1) 하에
#         허브·군집·교차 간선을 추가하여 수학적으로 Max Depth = 10을 정확히 보장합니다.
# ==============================================================================
import random

TEAM_POOL = [
    '안우현', '최우진', '배정우', '김서준', '이서아', '박도윤',
    '이가을', '김준석', '강석민', '김진우', '소준영', '김유진',
    '박채완', '이도현', '이대환', '오수아', '차재욱', '조형철',
    '정민서', '이준수', '김준성', '최하은', '정시우', '강유나',
    '윤여주', '김수인', '황정민', '김민정', '정다연', '안서윤',
    '권태현', '김주홍', '염도형', '김기헌', '박진성', '나희재',
]

def build_medium_100_graph(patient_zero, seed, hub_l1, hub_l2, local_edges, extra_edges, isolated_count=0):
    rng = random.Random(seed)
    reachable_count = 100 - isolated_count
    others = [p for p in TEAM_POOL if p != patient_zero] + [f"학생_{i:03d}" for i in range(37, 101 - isolated_count)]
    others = others[:reachable_count - 1]
    reachable_nodes = [patient_zero] + others
    isolated_nodes = [f"격리_{i:02d}" for i in range(1, isolated_count + 1)]

    g = {n: [] for n in reachable_nodes + isolated_nodes}
    depths = {patient_zero: 0}

    # 1. 깊이 10 뼈대 경로(Backbone) 구축 -> Max Depth = 10 보장
    backbone = [patient_zero] + others[:10]
    used = set(backbone)
    for i in range(10):
        g[backbone[i]].append(backbone[i + 1])
        depths[backbone[i + 1]] = i + 1

    # 2. 1단계(hub_l1), 2단계(hub_l2) 및 깊이 3~10 구간에 나머지 노드 분산 배치
    rem = [n for n in others if n not in used]
    cur = 0
    l1_nodes = [backbone[1]]
    for _ in range(hub_l1):
        node = rem[cur]; cur += 1
        g[patient_zero].append(node); depths[node] = 1; l1_nodes.append(node)
    for _ in range(hub_l2):
        node = rem[cur]; cur += 1
        g[rng.choice(l1_nodes)].append(node); depths[node] = 2

    eligible = [n for n, d in depths.items() if 2 <= d < 10]
    while cur < len(rem):
        node = rem[cur]; cur += 1
        parent = rng.choice(eligible)
        g[parent].append(node)
        depths[node] = depths[parent] + 1
        if depths[node] < 10:
            eligible.append(node)

    # 3. 최단 거리를 단축시키지 않는 선에서 군집 및 교차 간선 추가
    l1_all = [n for n, d in depths.items() if d == 1]
    l012_all = [n for n, d in depths.items() if d <= 2]
    added = 0
    while added < local_edges:
        u, v = rng.choice(l1_all), rng.choice(l012_all)
        if u != v and depths[v] <= depths[u] + 1 and v not in g[u]:
            g[u].append(v); added += 1

    added = 0
    while added < extra_edges:
        u, v = rng.sample(reachable_nodes, 2)
        if depths[v] <= depths[u] + 1 and v not in g[u]:
            g[u].append(v); added += 1

    # 4. 방역 격리 집단 내부 순환 연결 (외부 간선 0개)
    for i in range(isolated_count):
        u = isolated_nodes[i]
        g[u].append(isolated_nodes[(i + 1) % isolated_count])

    return g

def level_summary(result, max_depth):
    counts = [sum(1 for d, _ in result.values() if d == lv) for lv in range(1, max_depth + 1)]
    return ', '.join(f'{lv}단계: {c}명' for lv, c in enumerate(counts, 1) if c)

In [ ]:
# ==============================================================================
# [테스트 케이스 3 - 중1] 표준 커뮤니티 (노드 100개, Max Depth 10, 감염자: 이가을)
# - 역할: 깊이 10짜리 100명 그래프에서 2단계까지만 정확히 잘라내어(Cutoff) 멈추는지 검증
# ==============================================================================
g_med1 = build_medium_100_graph('이가을', seed=101, hub_l1=6, hub_l2=13, local_edges=14, extra_edges=48)
r_med1 = print_infection(g_med1, '이가을', max_depth=2, limit=5)
check(g_med1, '이가을', 2, r_med1)
print('(참고) 제한 없이 끝까지 퍼지면 최대 거리:', max(bfs_levels(g_med1, '이가을').values()))

[감염자: 이가을] 2단계 이내 감염 예상 인원: 21명 (본인 제외)

● 1단계 (7명)
  - 안우현    경로: 이가을 → 안우현
  - 김유진    경로: 이가을 → 김유진
  - 박채완    경로: 이가을 → 박채완
  - 이도현    경로: 이가을 → 이도현
  - 이대환    경로: 이가을 → 이대환
  ... 외 2명

● 2단계 (14명)
  - 최우진    경로: 이가을 → 안우현 → 최우진
  - 강유나    경로: 이가을 → 안우현 → 강유나
  - 황정민    경로: 이가을 → 안우현 → 황정민
  - 이준수    경로: 이가을 → 김유진 → 이준수
  - 정다연    경로: 이가을 → 김유진 → 정다연
  ... 외 9명
✅ 검증 통과 (감염자: 이가을, max_depth=2, 총 감염=21명)
(참고) 제한 없이 끝까지 퍼지면 최대 거리: 10


In [ ]:
# ==============================================================================
# [테스트 케이스 4 - 중2] 슈퍼 전파자 허브 (노드 100개, Max Depth 10, 감염자: 정민서)
# - 역할: 1단계 접촉자가 15명인 마당발 허브에서 2단계 만에 40명으로 급속 확산되는 상황 검증
# ==============================================================================
g_med2 = build_medium_100_graph('정민서', seed=202, hub_l1=14, hub_l2=24, local_edges=48, extra_edges=125)
r_med2 = print_infection(g_med2, '정민서', max_depth=2, limit=5)
check(g_med2, '정민서', 2, r_med2)
print('(참고) 제한 없이 끝까지 퍼지면 최대 거리:', max(bfs_levels(g_med2, '정민서').values()))

[감염자: 정민서] 2단계 이내 감염 예상 인원: 40명 (본인 제외)

● 1단계 (15명)
  - 안우현    경로: 정민서 → 안우현
  - 소준영    경로: 정민서 → 소준영
  - 김유진    경로: 정민서 → 김유진
  - 박채완    경로: 정민서 → 박채완
  - 이도현    경로: 정민서 → 이도현
  ... 외 10명

● 2단계 (25명)
  - 최우진    경로: 정민서 → 안우현 → 최우진
  - 나희재    경로: 정민서 → 안우현 → 나희재
  - 학생_041 경로: 정민서 → 안우현 → 학생_041
  - 김주홍    경로: 정민서 → 안우현 → 김주홍
  - 박진성    경로: 정민서 → 안우현 → 박진성
  ... 외 20명
✅ 검증 통과 (감염자: 정민서, max_depth=2, 총 감염=40명)
(참고) 제한 없이 끝까지 퍼지면 최대 거리: 10


In [ ]:
# ==============================================================================
# [테스트 케이스 5 - 중3] 고밀도 순환 + 방역 격리 12명 (노드 100개, Max Depth 10, 감염자: 권태현)
# - 역할: 2단계 추적(34명)과 10단계 끝까지 추적(87명)을 비교하며 방역 격리 12명 보호 효과 입증
# ==============================================================================
g_med3 = build_medium_100_graph('권태현', seed=303, hub_l1=11, hub_l2=21, local_edges=64, extra_edges=190, isolated_count=12)

# (1) 기본 2단계 이내 추적
r_med3_2step = print_infection(g_med3, '권태현', max_depth=2, limit=5)
check(g_med3, '권태현', 2, r_med3_2step)

# (2) 2단계 이상 끝까지(Depth 10) 추적 시에도 격리 집단 12명은 단 1명도 감염되지 않음
r_med3_full = infection_simulation(g_med3, '권태현', max_depth=10)
print('\n[끝까지(Depth 10) 추적 결과] 총 도달 감염자:', len(r_med3_full), '명 (본인 제외)')
print('레벨별 인원 →', level_summary(r_med3_full, 10))
check(g_med3, '권태현', 10, r_med3_full)
print('🛡️ 방역 격리 집단(격리_01 ~ 격리_12) 12명 끝까지 안전 보호 확인!')

[감염자: 권태현] 2단계 이내 감염 예상 인원: 34명 (본인 제외)

● 1단계 (12명)
  - 안우현    경로: 권태현 → 안우현
  - 소준영    경로: 권태현 → 소준영
  - 김유진    경로: 권태현 → 김유진
  - 박채완    경로: 권태현 → 박채완
  - 이도현    경로: 권태현 → 이도현
  ... 외 7명

● 2단계 (22명)
  - 최우진    경로: 권태현 → 안우현 → 최우진
  - 최하은    경로: 권태현 → 안우현 → 최하은
  - 학생_043 경로: 권태현 → 안우현 → 학생_043
  - 김민정    경로: 권태현 → 안우현 → 김민정
  - 학생_040 경로: 권태현 → 안우현 → 학생_040
  ... 외 17명
✅ 검증 통과 (감염자: 권태현, max_depth=2, 총 감염=34명)

[끝까지(Depth 10) 추적 결과] 총 도달 감염자: 87 명 (본인 제외)
레벨별 인원 → 1단계: 12명, 2단계: 22명, 3단계: 25명, 4단계: 12명, 5단계: 6명, 6단계: 2명, 7단계: 2명, 8단계: 2명, 9단계: 3명, 10단계: 1명
✅ 검증 통과 (감염자: 권태현, max_depth=10, 총 감염=87명)
🛡️ 방역 격리 집단(격리_01 ~ 격리_12) 12명 끝까지 안전 보호 확인!


In [ ]:
# ==============================================================================
# [테스트 케이스 6 - 예외 상황] 경계 조건 3종 검증
# ==============================================================================
# (1) 친구 리스트가 비어 있는 리프 노드('최우진')가 감염자 -> 아무도 감염되지 않음 (0명)
r_ex1 = print_infection(graph, '최우진', max_depth=2)
check(graph, '최우진', 2, r_ex1)

# (2) max_depth = 0 -> 감염자 본인만 존재하므로 전파 결과 0명
r_ex2 = infection_simulation(graph, '안우현', max_depth=0)
print('\nmax_depth=0 결과:', r_ex2)
check(graph, '안우현', 0, r_ex2)

# (3) 그래프에 존재하지 않는 이름 입력 -> ValueError 예외 발생 확인
try:
    infection_simulation(graph, '없는사람')
except ValueError as e:
    print('없는 이름 예외 처리 정상 동작:', e)

[감염자: 최우진] 2단계 이내 감염 예상 인원: 0명 (본인 제외)
✅ 검증 통과 (감염자: 최우진, max_depth=2, 총 감염=0명)

max_depth=0 결과: {}
✅ 검증 통과 (감염자: 안우현, max_depth=0, 총 감염=0명)
없는 이름 예외 처리 정상 동작: '없는사람'은(는) 그래프에 없는 사람입니다.
